# MACURA on a PyBullet quadrotor — study notebook

Reproduce and compare four model-based/model-free RL algorithms — **MACURA,
MBPO, M2AC, SAC** — on an unstable quadrotor hover task (`gym-pybullet-drones`).
All algorithm logic lives in the `.py` modules; this notebook only imports, runs,
narrates, and plots. Everything saves to Google Drive. Set **`SMOKE=True`** for a
quick top-to-bottom pass; `False` for the ~20k-step run.

Sections: **2** setup · **3** explore the env (with a clear drone video) · **4**
the four algorithms · **5** exploration noise · **6** training (~20k) · **7**
evaluate best models + videos · **8** results comparison.

## 2. Imports & setup
Mount Drive, require a GPU, set headless rendering, clone+install, then print
versions and load the YAML config. Look for `GPU: ...` and the version line.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
os.environ['MUJOCO_GL'] = 'egl'
warnings.filterwarnings('ignore')
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
os.environ['GITHUB_TOKEN'] = TOKEN
print('token loaded')

In [ ]:
%%bash
set -e
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/silvergjeka22/macura-drone/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')
import copy, json, yaml, numpy as np
import pybullet, gymnasium, stable_baselines3
PYB_VER = getattr(pybullet, '__version__', None) or pybullet.getAPIVersion()
print('torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER, '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)

from envs import drone_env
from viz import plots
from training import train as trainer
from algorithms import sac as sac_mod
from models import ensemble as ens

CFG = yaml.safe_load(open('/content/macura-drone/configs/macura_drone.yaml'))
DRIVE = CFG['experiment']['drive_root']
SMOKE = True   # True = tiny quick pass; False = ~20k-step run

def make_cfg(smoke=False, fast=True):
    c = copy.deepcopy(CFG)
    if fast:
        for k, v in c.get('profiles', {}).get('fast', {}).items():
            c[k].update(v) if isinstance(v, dict) else c.__setitem__(k, v)
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=500, eval_episodes=2, seeds=[0])
        c['rollout'].update(num_rollouts=100, freq_steps=250)
    c['env']['backend'] = 'pybullet'
    c['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
    return c

cfg = make_cfg(smoke=SMOKE, fast=True)
print('backend', cfg['env']['backend'], '| steps', cfg['experiment']['total_env_steps'],
      '| seeds', cfg['experiment']['seeds'])

## 3. Explore the environment

**Task.** A Crazyflie quadrotor must take off and hover at `(0, 0, 1 m)`. The
observation is a 13-dim kinematic state `[pos_err, quat, lin_vel, ang_vel]`; the
action is 4 normalized rotor commands around hover RPM; the reward is a dense
quadratic on pose/velocity/tilt/effort.

**Difference from the paper.** MACURA was validated on MuJoCo *locomotion*
(stable systems). A quadrotor is **unstable** — it balances on thrust, so model
errors compound within a few steps. That is exactly the regime where adaptive
rollout truncation should matter, which is why it is a harder, more revealing test.

First, create the env.

In [ ]:
env, obs_dim, act_dim = drone_env.make_env(cfg['env'], seed=0, render=True)
obs0, _ = env.reset(seed=0)
print('obs_dim', obs_dim, '| act_dim', act_dim, '| HOVER_RPM', round(env.HOVER_RPM, 1))

A high-resolution video of the drone (camera follows it and zooms in, so the
quadrotor is clearly visible). Saved to Drive and shown inline.

In [ ]:
vid = f'{DRIVE}/videos/env_demo.mp4'
plots.record_env_video(env, policy='random', num_steps=120, save_path=vid, fps=30)
from IPython.display import Video, display
display(Video(vid, embed=True, width=520))

Behaviour study: random actions (the drone tumbles fast — the instability) vs a
constant zero-thrust action (it simply drops).

In [ ]:
rec_rand = plots.collect_rollout(env, policy='random', num_steps=200, seed=1)
rec_hover = plots.collect_rollout(env, policy='hover', num_steps=200, seed=1)
plots.plot_rollout_traces(rec_rand, title='Random actions', save_path=f'{DRIVE}/plots/s3_random.png')
plots.plot_rollout_traces(rec_hover, title='Zero thrust', save_path=f'{DRIVE}/plots/s3_hover.png');

State distribution the model must learn, and the dense reward landscape (smooth
and peaked at the target — easy for the ensemble to predict).

In [ ]:
plots.plot_state_distributions(env, num_episodes=10, num_steps=60, save_path=f'{DRIVE}/plots/s3_state.png')
plots.plot_reward_landscape(cfg, save_path=f'{DRIVE}/plots/s3_reward.png');

## 4. The four algorithms

All four share the **same SAC backbone** and (the model-based ones) the **same
probabilistic ensemble** — only the rollout differs:

- **MACURA** — model-based; rollout length is *adaptive*: keep imagining while the
  ensemble agrees (uncertainty `u_GJS < kappa`), truncate the moment it doesn't.
- **MBPO** — model-based; *fixed* truncated-linear rollout length, no uncertainty.
- **M2AC** — model-based; fixed length, then *masks out* the least-trustworthy
  imagined steps.
- **SAC** — model-free; *no* model rollouts at all (the reference baseline).

Below we instantiate the shared backbone (SAC + ensemble) that every algorithm
uses, then show a schematic of the four rollout strategies.

In [ ]:
agent = sac_mod.build_sac(obs_dim, act_dim, {**cfg['sac'], 'buffer_size': 10000}, 'cuda', 0)
ensemble = ens.build_ensemble(cfg['ensemble'], obs_dim, act_dim, 'cuda')
n_sac = sum(p.numel() for p in agent.policy.parameters())
n_ens = sum(p.numel() for p in ensemble['model'].parameters())
print(f'shared SAC policy params: {n_sac:,}  |  ensemble params: {n_ens:,}'
      f'  |  ensemble members: {len(ensemble["model"].members)}')
print('builders: algorithms.{macura,mbpo,m2ac,sac}.build_* wrap these same pieces')
plots.plot_rollout_strategy_schematic(save_path=f'{DRIVE}/plots/s4_strategy.png');

## 5. Exploration noise

How the agent explores while collecting real data:
- **Pink noise** (`1/f`, temporally *correlated*) — smooth, momentum-like motions
  that suit continuous drone control; this is what the paper uses for MACURA and
  what we use here (`exploration.type: pink_noise`, real FFT generator in
  `training/train.py:_PinkNoise`). Applied to **all four** algorithms for fairness.
- **White noise** — uncorrelated, jittery; tends to cancel out and explore less.
- **Init noise** (`env.init_*_noise`) — always on: episodes start near hover with
  small random perturbations so competence is reachable quickly.
- **Sensor noise** (`env.noise`) — off by default; only for robustness studies.

The plot shows pink vs white as a time series and their power spectra — pink has
the characteristic downward `1/f` slope.

In [ ]:
plots.plot_noise_comparison(save_path=f'{DRIVE}/plots/s5_noise.png');

## 6. Training (fast, ~20k steps)

Train all four with the **fast** profile (trimmed UTD / rollouts / ensemble
epochs; ~20k steps when `SMOKE=False`). Evaluation is **periodic** (not per step),
and only the **best model per algorithm/seed** is saved to Drive
(`checkpoints/<algo>_seed<seed>_best.zip`). Runs are idempotent — an existing log
is reused. Watch the printed `return ... <- best` lines.

In [ ]:
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        lp = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(lp):
            print('skip (exists):', algo, seed); runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))
print('done:', len(runs), 'runs')

Training progress: evaluation return vs real steps, and crash rate while learning.

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/s6_sample_eff.png')
plots.plot_failure_rate(runs, save_path=f'{DRIVE}/plots/s6_failure.png');

## 7. Evaluate best models + video

Load each best checkpoint from Drive, run a deterministic evaluation, and save an
`.mp4` per algorithm/seed to Drive — then display them inline.

In [ ]:
eval_metrics = trainer.evaluate_best(cfg, DRIVE, device='cuda')
import pandas as pd
display(pd.DataFrame(eval_metrics))
video_paths = trainer.record_best_videos(cfg, DRIVE, device='cuda')
for name, path in video_paths.items():
    print(name, '->', path)
    display(Video(path, embed=True, width=480))

## 8. Results comparison

Sample efficiency (mean ± std over seeds), failure rate, final-policy quality, and
the MACURA signature (rollout length & kappa over training). With `SMOKE=True`
the curves are too short to be conclusive — they validate the pipeline; run
`SMOKE=False` for the real comparison.

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/s8_sample_eff.png')
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/s8_final_quality.png')
mac = next((r for r in runs if r['algo'] == 'macura'), None)
if mac:
    plots.plot_rollout_depth(mac, save_path=f'{DRIVE}/plots/s8_rollout_depth.png')
import pandas as pd
display(pd.DataFrame([{'algo': r['algo'], 'seed': r['seed'],
                       'best_return': round(r.get('best_return', 0.0), 1)} for r in runs]))

**Discussion (fill in after a `SMOKE=False` run).** Compare: does the model help
(MACURA/MBPO/M2AC vs SAC)? Does adaptivity help (MACURA vs MBPO — does fixed-
horizon MBPO wobble on the unstable drone)? Is adaptive truncation better than
masking (MACURA vs M2AC)? State the winner and why, honestly — a mixed result on
the drone is still a valid finding.